In [ ]:
import numpy as np
import math, random
import pandas as pd
from scipy.spatial.transform import Rotation as rotate
from scipy.stats import truncnorm

from functions.representation import find_center_point_LWLC, load_objects
from functions.vectors import cosine_similarity, find_axis_of_rotation_geon_only, find_axis_of_rotation_geon_and_spatcon, same_object, calculate_axis_difficulty
from functions.plotting import add_frame, prepare_rotation_graphs

import plotly.graph_objects as go
import plotly.offline as pyo
import copy

# Initialize Plotly for offline mode in Jupyter Notebook
pyo.init_notebook_mode(connected=True)

# functions

def normal_dist_w_limits(low, high, mean, std):
    a = (low - mean) / std
    b = (high - mean) / std

    return truncnorm.rvs(a, b, loc=mean, scale=std)

def calculate_step_size_simple(angular_disparity, speed_constant):
    step_size = speed_constant * math.sqrt(angular_disparity)

    return step_size

# Experiment Variables

In [ ]:
show_animations = False
gender_choice = "random"            # can be 'random' or 'even'
gender_effect_applied = False
gk_test = False
jj_test = True

if gk_test or show_animations:
    object_file = "experimental_data/GanisKievetShapes.txt"
    rotation_list = [0, 50, 100, 150]
    axis_list = [np.array([0, 1, 0])]

if jj_test:
    object_file = "experimental_data/JostJansenShapes.txt"
    rotation_list = [0, 45, 90, 135, 180]
    axis_list = [np.array([0, 1, 0]), np.array([0, 0, 1])]

mirrored_list = [True, False]
gender_list = [0, 1]                        # 0 = f, 1 = m
objects = load_objects(object_file)

if gk_test:
    objects = objects * 2
    n = 50

if jj_test:
    objects = objects * 28
    n = 40

if show_animations:
    objects = [objects[0]]          # change index if you want a different shape (0-47), or make your own object
    n = 1

average_run_time = 0
false_positives = 0
false_negatives = 0
m_counter = 0
f_counter = 0

csv_data = {
    "subject_num": [],
    "gender": [],
    "angle": [],
    "time": [],
    "response": [],
    "expected_answer": [],
    "correct": [],
    "false_positive": [],
    "false_negative": [],
    "source": [],

    "axis": [],
    "axis_difficulty": [],
    "num_of_repeats": [],
    "encoding_time": [],
    "landmarking_time": [],
    "rotation_time": [],
    "decision_time": []
}
print(len(objects))

# Main Loop

In [ ]:
# ----------------- #
#   SUBJECT LOOP    #
# ----------------- #

subject_num = 0
for subject in range(n):

    # SUBJECT VARIABLES:

    subject_num += 1

    if gender_choice == "random":
        gender = random.choice(gender_list)
    else:                                           # even gender count in subjects
        if subject_num <= n/2:
            gender = 0
        else:
            gender = 1

    # gender count
    if gender == 0:
        f_counter += 1
    else:
        m_counter += 1

    # timing
    production_time = 50                                                                            # add distribution?
    propositional_difficulty_time = None                                                            # determined later
    object_encoding_time = normal_dist_w_limits(low=200, high=700, mean=450, std=100)               # for TWO objects 
    initial_axis_id_time = normal_dist_w_limits(low=100, high=200, mean=150, std=25)               
    response_selection_time = normal_dist_w_limits(low=150, high=300, mean=225, std=50)             # i.e. deciding what button to press
    response_motor_time = normal_dist_w_limits(low=100, high=250, mean=175, std=50)                 # i.e. pressing response button

    # rotation
    speed_constant = None                                                                           # determined later
    speed_constant_decrease = 0.5                                                                   # add distribution?

    # decision
    if gender_effect_applied:
        repeat_threshold = round(normal_dist_w_limits(low=1, high=4, mean=1.5 + (1-gender), std=1))     # 0-4 repeated steps
    else:
        repeat_threshold = round(normal_dist_w_limits(low=0, high=4, mean=1.5, std=0.5))                # 0-4 repeated steps
        
    wrong_guess_decrease = 0.15                                                                         # add distribution affected by gender/axis difficulty?
    effect_of_anglular_disp_on_wrong_guess = 0.1
    post_rotation_prop_diff_weight = 0.75                                                                # how much prop difficulty value remians after rotation

    # similarity thresholds
    geon_alignment_threshold = normal_dist_w_limits(low=0.6, high=1, mean=0.8, std=0.1)                 # cosine similarity threshold for landmark geon alignment before phase two starts
    landmark_alignment_threshold = normal_dist_w_limits(low=0.9, high=1, mean=0.99, std=0.01)           # cosine similarity threshold for landmark geon and spat con alignment during phase two
    object_alignment_threshold = normal_dist_w_limits(low=0.9, high=1, mean=0.98, std=0.01)             # final check similarity threshold

    # --------------------- #
    #   TEST OBJECT LOOP    #
    # --------------------- #

    for object in objects:

        # ROTATION VARIABLES:

        total_run_time = 0
        encoding_time = 0
        landmarking_time = 0
        rotation_time = 0
        decision_time = 0
        wrong_guess_chance = 0.5                                         # starts at 50%
        angle = random.choice(rotation_list)
        axis = random.choice(axis_list)
        mirrored = random.choice(mirrored_list)
        axis_difficulty = calculate_axis_difficulty(axis)
        r = rotate.from_rotvec(np.deg2rad(angle) * axis)
        center_point = np.array([0,0,0])

        # REMAINING SUBJECT VARIABLES (affected by axis difficulty):

        propositional_difficulty_time = normal_dist_w_limits(low=0.5, high=1.5, mean=1 + (0.25 * axis_difficulty), std=0.25)              # more difficult on combination rotations
        
        if gender_effect_applied:
            speed_constant = normal_dist_w_limits(low=1, high=5, mean=3 + (0.5 * gender) - axis_difficulty, std=1)                      # idek like 2-4 seems realistic? maybe reduce on repeat --> slower start on combination rotations
        else:
            speed_constant = normal_dist_w_limits(low=1, high=5, mean=3 - axis_difficulty, std=1)

        # MAKE ORIGINAL AND TARGET OBJECTS:

        original_object = copy.deepcopy(object)
        target_object = copy.deepcopy(object)
        target_object.rotate(r)
        if mirrored:
            target_object.flip_last_geon()

        # make copy of original object
        original_object_reset = copy.deepcopy(original_object)

        # CALCULATE ROTATION DATA:
        axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point)
        total_axis, _, total_angular_disparity = find_axis_of_rotation_geon_and_spatcon(original_object, target_object, center_coords=center_point)

        # TIMING:

        # add time needed for encoding step
        encoding_time += object_encoding_time
        total_run_time += encoding_time
        
        # add time needed for landmarking step
        landmarking_time += production_time                                                                                 # check geon
        landmarking_time += production_time + (total_angular_disparity * propositional_difficulty_time)                     # check spatial connection
        total_run_time += landmarking_time

        # add time for initial axis calculation
        rotation_time += initial_axis_id_time
        total_run_time += initial_axis_id_time

        # ----------------- #
        #   ROTATON LOOP    #
        # ----------------- #

        repeat_count = 0
        same = False
        decision = None
        while not same and repeat_count <= repeat_threshold:

            # prepare animation graphs
            if show_animations:
                axis_animation_fig, overlap_animation_fig, sidebyside_animation_fig = prepare_rotation_graphs(original_object, target_object, axis_of_rotation, center_point, production_time)
                axis_animation = []
                overlap_animation = []
                sidebyside_animation = []

            # set landmark vectors, angular disparity
            original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
            target_landmark_geon_vector = target_object.get_landmark_geon().get_vector()
            original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()
            target_spatcon_direction = target_object.get_landmark_spatial_connection().get_vector()
            # total_angular_disparity_part2 = None
            prev_step_size = None

            # PHASE ONE: GEON ALIGNMENT

            loop_count = 0
            while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < geon_alignment_threshold:     # checking cosine similarity between target and goal geons

                # find best axis/direction of rotation, angular disparity
                axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_angle=curr_step_angular_disparity, total_angular_disparity=total_angular_disparity)

                # calculate step size based on angular disparity and rotation speed
                step_size = calculate_step_size_simple(curr_step_angular_disparity, speed_constant)

                if prev_step_size != None and prev_step_size < step_size:
                    break
                    # failed rotation
                else:
                    prev_step_size = step_size

                # apply rotation to original object
                r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis
                original_object.rotate(r)

                # update original geon and spatial connection vectors
                original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
                original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()

                # add animation frames to graphs
                if show_animations:
                    add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)
                    original_centerpoint_vec = find_center_point_LWLC(original_object)
                    copy_og_obj = copy.deepcopy(original_object)
                    copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
                    add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)
                    add_frame(sidebyside_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

                # add rotation time
                total_run_time += production_time * 3
                rotation_time += production_time * 3

                # count loop
                loop_count += 1

                # emergency break
                if loop_count > 100:
                    break

            # PHASE TWO: FULL ALIGNMENT

            prev_step_size = None
            loop_count = 0
            while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < landmark_alignment_threshold or cosine_similarity(original_spatcon_direction, target_spatcon_direction) < landmark_alignment_threshold:     # checking cosine similarity between target and goal geons and spatial connections

                # find best axis/direction of rotation, angular disparity
                axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_and_spatcon(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_angle=curr_step_angular_disparity, total_angular_disparity=total_angular_disparity)

                # calculate step size based on angular disparity and rotation speed
                step_size = calculate_step_size_simple(curr_step_angular_disparity, speed_constant)

                if prev_step_size != None and prev_step_size < step_size:
                    break
                    # failed rotation
                else:
                    prev_step_size = step_size

                # apply rotation to original object
                r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis
                original_object.rotate(r)

                # update original geon and spatial connection vectors
                original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
                original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()

                # add animation frames to graphs
                if show_animations:
                    add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)
                    original_centerpoint_vec = find_center_point_LWLC(original_object)
                    copy_og_obj = copy.deepcopy(original_object)
                    copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
                    add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)
                    add_frame(sidebyside_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

                # add rotation time
                total_run_time += production_time * 3
                rotation_time += production_time * 3

                # count loop
                loop_count += 1

                # emergency break
                if loop_count > 100:
                    break

            # show animations
            if show_animations:
                axis_animation_fig.frames = axis_animation
                axis_animation_fig.show()
                overlap_animation_fig.frames = overlap_animation
                overlap_animation_fig.show()
                sidebyside_animation_fig.frames = sidebyside_animation
                sidebyside_animation_fig.show()

            # check overall similarity
            same, similarity_check_run_time = same_object(original_object, target_object, object_alignment_threshold, total_angular_disparity, production_time, propositional_difficulty_time, post_rotation_prop_diff_weight)
            decision_time += similarity_check_run_time
            total_run_time += similarity_check_run_time

            if not same:

                # reset original object
                original_object = original_object_reset
                axis_of_rotation, direction, total_angular_disparity = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point)
                speed_constant = speed_constant - speed_constant_decrease
                wrong_guess_chance = wrong_guess_chance - wrong_guess_decrease
                repeat_count += 1

                # reset time
                total_run_time += production_time

                # print("REPEAT #" + str(repeat_count))
                # print("\nRUN TIME:\n" + str(round(total_run_time/1000, 3)) + " seconds")
                # print("speed_constant: " + str(speed_constant))

        # decision
        if same:
            decision = "same"
        else:
            if random.uniform(0, 1) <= wrong_guess_chance + effect_of_anglular_disp_on_wrong_guess*(total_angular_disparity/180):              # chance that wrong guess occurs
                decision = "same"
            else:
                decision = "different"

        # add timing
        decision_time += response_selection_time + response_motor_time
        total_run_time += response_selection_time + response_motor_time

        # count errors
        if mirrored and decision == "same":
            false_positives += 1
        elif not mirrored and decision == "different":
            false_negatives += 1

        # add to avg run time variable
        average_run_time += round(total_run_time/1000, 3)

        # print()
        # print("propositional_difficulty_time: " + str(propositional_difficulty_time))
        # print("object_encoding_time: " + str(object_encoding_time))
        # print("speed_constant: " + str(speed_constant))
        # print("repeat_threshold: " + str(repeat_threshold))
        # print("geon_alignment_threshold: " + str(geon_alignment_threshold))
        # print("landmark_angle_threshold: " + str(landmark_angle_threshold))
        # print("object_angle_threshold: " + str(object_angle_threshold))

        # add trial data to csv
        csv_data['subject_num'].append(subject_num)
        csv_data['gender'].append("female" if gender == 0 else "male")
        csv_data['angle'].append(angle)
        csv_data['time'].append(round(total_run_time/1000, 3))
        csv_data['response'].append(decision)
        csv_data['expected_answer'].append("different" if mirrored else "same")
        csv_data['correct'].append(1 if decision == csv_data['expected_answer'][-1] else 0)
        csv_data['false_positive'].append(1 if mirrored and decision == "same" else 0)
        csv_data['false_negative'].append(1 if not mirrored and decision == "different" else 0)
        csv_data['source'].append("model")

        csv_data['axis'].append(axis)
        csv_data['axis_difficulty'].append(axis_difficulty)
        csv_data['num_of_repeats'].append(repeat_count)
        csv_data['encoding_time'].append(round(encoding_time/1000, 3))
        csv_data['landmarking_time'].append(round(landmarking_time/1000, 3))
        csv_data['rotation_time'].append(round(rotation_time/1000, 3))
        csv_data['decision_time'].append(round(decision_time/1000, 3))

print()
print("total runs: " + str(n*len(objects)))
print("average_run_time: " + str(average_run_time/(n*len(objects))))
print("false_positives: " + str(false_positives))
print("false_negatives: " + str(false_negatives))
print("num of boys: " + str(m_counter))
print("num of girls: " + str(f_counter))

# save experiment data to csv
if gk_test:
    name = 'rotation_model_data_gk.csv'
elif jj_test:
    name = 'rotation_model_data_jj.csv'
else:
    name = 'rotation_model_data.csv'
    
df = pd.DataFrame(csv_data)
df.to_csv(name, index=False)